# Week 7: How Canadian is the Giller Prize?

This week we will use our Giller data set to ask two questions:

1. **How Canadian are the nominees?** Where were they born, and where do they live now?
2. **Where in Canada do the nominees come from?** Which provinces were they born in, and how does that compare with the number of people who live in each province?

## 1. Loading the clean data

We'll start from the file `giller_nominees_clean.csv` that we saved last week, with the stray spaces already removed. It is in the same folder as this notebook.

In [ ]:
import csv

with open("giller_nominees_clean.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    nominees = list(reader)

print(len(nominees), "nominated books")

## 2. Where were the nominees born?

### Step 1: One birthplace per author

First we build a dictionary that maps each author to their birthplace.

In [ ]:
# Create an empty dictionary 
birthplaces = {}

# For each row in the list of nominees
for book in nominees:
    
    # Get the birth place and name of the author of this book
    born = book["Born"]  
    author = book["Author"]

    # Add the birthplace of the author to the dictionary
    birthplaces[author] = born

# How many unique authors are in the data set?
print(len(birthplaces), "authors")
print(birthplaces["Shyam Selvadurai"])

### Step 2: Splitting city from region

Looking at this column we see values like: 
- "Brampton, ON"
- "London, England"
- "Baltimore, MD, USA"
- "South Africa"
- "Ontario"

How can we describe region from values like this?  How can we extract only the regions from these strings?


In [ ]:
born = birthplaces["Shyam Selvadurai"]

# divide the birthplace into its parts
# parts = ??

print(parts)

Notice the space at the start of `' Sri Lanka'`: we'll need `.strip()` again. Also, some birthplaces have more than one comma, and some have none at all:

In [ ]:
print(birthplaces["Joseph Boyden"].split(","))
print(birthplaces["Timothy Taylor"].split(","))

Using index `-1` always gives us the **last** piece, which is the region we want, no matter how many commas there are. And if there's no comma, the whole value is the region.

Let's put these steps together with an example value for `place`.

In [ ]:
# The region is the last item in the list, and we want to remove
# the whitespace around it.
region = parts[-1].strip()
print(region)

Now that we know how to obtain the region from a birth place, let's make it a function so that we can reuse this code.

In [ ]:
def get_region(place):
    """ Return the province, state or country at the end of place.

        Example: get_region("Colombo, Sri Lanka") returns "Sri Lanka"
    """
    parts = place.split(",")   
    region = parts[-1].strip()
    
    return region


print(get_region("Colombo, Sri Lanka"))
print(get_region("Willowdale, North York, ON"))
print(get_region("Venezuela"))

### Step 3: Count the regions, and look at what we got

The pattern repeats.  To find out how many authors were born in each region we need to count them. 

Let's make another dictionary, and use our `get_region()` function

This is a loop pattern we've seen before and a very common one!

In [ ]:
region_counts = {}



for region in region_counts:
    print(region, region_counts[region])

Decisions to make:

- **The same place written different ways:** 
    - `Ontario` and `ON`, `Alberta` and `AB`, `British Columbia` and `BC`. 
- **Mixed levels:** 
    - provices vs countries 
- **Missing detail:** 
    - one birthplace is just `Canada`.
- **Historical names:** 
    - `West Germany`, `South Vietnam`, and `Yugoslavia` 
- **Placeholders:** a `-` means "unknown." (We'll skip these when we count.)

### Step 4: Fixing values with a dictionary

Big idea: record the changes we want to make in a dictionary

In [ ]:
corrections = {
    # Full province names: use the abbreviations found everywhere else
    "Ontario": "ON",
    "Alberta": "AB",
    "British Columbia": "BC",
}

# Trying out a correction
region = "Ontario"
if region in corrections:
    region = corrections[region]
print(region)

### Step 5: Clean each birthplace once

Let's save our cleaned data into a dictionary:

In [ ]:
born_regions = {}

for author in birthplaces:
    region = get_region(birthplaces[author])
    if region in corrections:
        region = corrections[region]
    born_regions[author] = region

print(birthplaces["Shyam Selvadurai"], "->", born_regions["Shyam Selvadurai"])
print(birthplaces["Loghan Paylor"], "->", born_regions["Loghan Paylor"])

In [ ]:
# Now we can count:
born_region_counts = {}

for author in born_regions:
    region = born_regions[author]

    if region == "-":
        print(f"Skipping {author}: birthplace unknown")
    elif region in born_region_counts:
        born_region_counts[region] = born_region_counts[region] + 1
    else:
        born_region_counts[region] = 1

print()
for region in born_region_counts:
    print(region, born_region_counts[region])

### Step 6: Born in Canada, or elsewhere?

Finally, let's group the regions into two categories. 

In [ ]:
canadian_regions = ["AB", "BC", "MB", "NB", "NL", "NS", "NT", "NU", "ON", "PE", "QC", "SK", "YT", "Canada", "Treaty 6 Territory"]

born_in_canada = 0
born_elsewhere = 0

for author in born_regions:




total = born_in_canada + born_elsewhere
print(f"Born in Canada:    {born_in_canada} ({born_in_canada / total * 100:.0f}%)")
print(f"Born elsewhere:    {born_elsewhere} ({born_elsewhere / total * 100:.0f}%)")

## 3. Where do the nominees live now?

We'll follow the same steps we used for birthplaces, starting with one residence per author.

In [ ]:
residences = {}

for book in nominees:
    residence = book["Residence"]
    if residence != "":
        residences[book["Author"]] = residence

print(len(residences), "authors with a residence")
print(residences["Shani Mootoo"])

### Step 1: Look before you count

Before we count anything, let's find out what is different.

In [ ]:
for author in residences:
    if " and " in residences[author]:
        print(f"{author}: {residences[author]}")


Some people have multiple residences.

Let's use `.split()` to get the first residence.

In [ ]:
place = "Montreal, QC and Paris, France"
parts = place.split(" and ")
print(parts)
print(parts[0])

place = "Toronto, ON"
parts = place.split(" and ")
print(parts[0])   # only one place, so nothing changes

### Step 2: Look at the regions

Just as we did for birthplaces, let's count the regions before cleaning them, so that we can see what problems they contain. For each author we take the first place listed, then use `get_region()`.

In [ ]:
region_counts = {}

for author in residences:
    parts = residences[author].split(" and ")
    region = get_region(parts[0])
    if region in region_counts:
        region_counts[region] = region_counts[region] + 1
    else:
        region_counts[region] = 1

for region in region_counts:
    print(region, region_counts[region])

### Step 3: Adding to our corrections

There are additional problems that we need to make corrections for.  So we will add
them to our `corrections` dictionary.

In [ ]:
# Full names: use the abbreviations found everywhere else
corrections["New Brunswick"] = "NB"
corrections["Newfoundland"] = "NL"
corrections["Nova Scotia"] = "NS"
corrections["Quebec"] = "QC"
corrections["South-Eastern Ontario"] = "ON"

# The United States, written out in full
corrections["United States"] = "USA"

# Leftovers from the "first place" rule
corrections["Galiano Island"] = "BC"   # an island in BC

print(len(corrections), "corrections")

### Step 4: Clean each residence once

Now we build `home_regions` in the same way we built `born_regions`, using the first place listed. We added the new corrections *before* running this loop, so they are used.

In [ ]:
home_regions = {}

for author in residences:
    parts = residences[author].split(" and ")
    region = get_region(parts[0])
    if region in corrections:
        region = corrections[region]
    home_regions[author] = region

print(residences["Michael Winter"], "->", home_regions["Michael Winter"])

In [ ]:
# Now we can count
home_region_counts = {}

for author in home_regions:
    region = home_regions[author]

    if region == "-":
        print(f"Skipping {author}: residence unknown")
    elif region in home_region_counts:
        home_region_counts[region] = home_region_counts[region] + 1
    else:
        home_region_counts[region] = 1

print()
for region in home_region_counts:
    print(region, home_region_counts[region])

### Step 5: Living in Canada, or elsewhere?

This is the same calculation we did for birthplaces, using the same `canadian_regions` list.

In [ ]:
lives_in_canada = 0
lives_elsewhere = 0

for author in home_regions:
    region = home_regions[author]
    if region in canadian_regions:
        lives_in_canada = lives_in_canada + 1
    elif region != "-":   # skip unknown residences
        lives_elsewhere = lives_elsewhere + 1

total = lives_in_canada + lives_elsewhere


In [ ]:
print(f"Lives in Canada:   {lives_in_canada} ({lives_in_canada / total * 100:.0f}%)")
print(f"Lives elsewhere:   {lives_elsewhere} ({lives_elsewhere / total * 100:.0f}%)")

## 4. How Canadian? Putting birth and residence together

Every author falls into one of four groups:

|                      | **Lives in Canada**       | **Lives elsewhere**        |
|----------------------|---------------------------|----------------------------|
| **Born in Canada**   | born here, lives here     | born here, lives elsewhere |
| **Born elsewhere**   | born elsewhere, lives here| born elsewhere, lives elsewhere |

Let's sort authors into each of these groups

In [ ]:
born_here_lives_here = 0
born_here_lives_elsewhere = 0
born_elsewhere_lives_here = 0
born_elsewhere_lives_elsewhere = []   # a list, so that we can look at the names

for author in born_regions:
    # check this author has a residence too, so we don't get a KeyError
    if author in home_regions:
        born = born_regions[author]
        home = home_regions[author]

        if born == "-" or home == "-":
            print(f"Skipping {author}: birthplace or residence unknown")
        elif born in canadian_regions and home in canadian_regions:
            born_here_lives_here = born_here_lives_here + 1
        elif born in canadian_regions and home not in canadian_regions:
            born_here_lives_elsewhere = born_here_lives_elsewhere + 1
        elif born not in canadian_regions and home in canadian_regions:
            born_elsewhere_lives_here = born_elsewhere_lives_here + 1
        else:
            # the only group left: born elsewhere and lives elsewhere
            born_elsewhere_lives_elsewhere.append(author)

neither = len(born_elsewhere_lives_elsewhere)
total = born_here_lives_here + born_here_lives_elsewhere + born_elsewhere_lives_here + neither

In [ ]:

print(f"Born in Canada, lives in Canada:   {born_here_lives_here} ({born_here_lives_here / total * 100:.0f}%)")
print(f"Born in Canada, lives elsewhere:   {born_here_lives_elsewhere} ({born_here_lives_elsewhere / total * 100:.0f}%)")
print(f"Born elsewhere, lives in Canada:   {born_elsewhere_lives_here} ({born_elsewhere_lives_here / total * 100:.0f}%)")
print(f"Born elsewhere, lives elsewhere:   {neither} ({neither / total * 100:.0f}%)")

Who are the authors in the last group?

In [ ]:
for author in born_elsewhere_lives_elsewhere:
    print(f"{author}: born {birthplaces[author]}; lives {residences[author]}")

## 5. Which provinces were the nominees born in?

In [ ]:
province_populations = {
    "BC": 5000879,
    "AB": 4262635,
    "SK": 1132505,
    "MB": 1342153,
    "ON": 14223942,
    "QC": 8501833,
    "NB": 775610,
    "NS": 969383,
    "PE": 154331,
    "NL": 510550,
    "YT": 40232,
    "NT": 41070,
    "NU": 36858,
}

### Step 2: Counting nominees by province of birth

This is our usual counting pattern, with two small changes:

- We start every province's count at `0`, so that a province with no nominees still shows up in our results.
- We only count an author if their region is a key in `born_province_counts`. 

In [ ]:
born_province_counts = {}

# Start every province at 0
for province in province_populations:
    born_province_counts[province] = 0

# Count the nominees born in each province
for author in born_regions:
    region = born_regions[author]
    if region in born_province_counts:
        born_province_counts[region] = born_province_counts[region] + 1

# Add up the counts to get the number of nominees we counted
born_in_a_province = 0
for province in born_province_counts:
    born_in_a_province = born_in_a_province + born_province_counts[province]

print(born_in_a_province, "nominees born in a province or territory")


In [ ]:

for province in born_province_counts:
    print(province, born_province_counts[province])

### Step 3: Comparing shares

A fairer comparison asks: what **share** of the nominees was born in each province, and what share of Canadians lives there? 

In [ ]:
for province in province_populations:
    nominee_share = born_province_counts[province] / born_in_a_province * 100
    population_share = province_populations[province] / canada_population * 100
    print(f"{province}: {nominee_share:4.1f}% of nominees, {population_share:4.1f}% of Canadians")

### Step 4: Nominees per million people

Another common way to compare places of different sizes is a **rate**: how many nominees there are for every million people who live there.

In [ ]:
for province in province_populations:
    per_million = born_province_counts[province] / province_populations[province] * 1000000
    print(f"{province}: {born_province_counts[province]:2} nominees, {per_million:4.1f} per million people")

Next week we will show how to create a graph like this one, but for now we can enjoy the output.

![Two maps of Canada shading each province by Giller nominees per million people, for where nominees were born and where they live now. Newfoundland and Labrador is darkest on both.](Nominees_by_province.png)

**Discussion:**
- Newfoundland and Labrador produces far more nominees than its population would suggest. What might explain that? What else would you want to know before making a claim about it?
- Quebec produces fewer nominees than its population would suggest.
- Prince Edward Island and the territories have only 0 or 1 nominees each. How many nominees do you think a province needs before its percentage or rate tells us something?
- We compared birthplaces with **today's** populations, but most nominees were born decades ago, when the provinces' shares of Canada's population were different. How might that change the comparison?

## 6. Saving our cleaned table

We've made a lot of decisions this week: one row per author, the "first place listed" rule, the corrections dictionary, and which regions count as Canada. Let's save the results as a new CSV file, so that we (and other people) can use the cleaned data without redoing all of this work. Just as in Week 6, the code is a record of exactly what we changed.

Our new table will have **one row per author**, with these columns:

| Column | What it holds |
|---|---|
| `Author` | the author's name |
| `Born` | the birthplace, as recorded in the spreadsheet |
| `Born region` | the cleaned province, state, or country |
| `Born in Canada` | `Canada`, `Elsewhere`, or `Unknown` |
| `Residence` | the first place listed in the `Residence` column |
| `Lives region` | the cleaned province, state, or country |
| `Lives in Canada` | `Canada`, `Elsewhere`, or `Unknown` |
| `Nominations` | how many times the author was nominated |

We use the words `Unknown` instead of `"-"`, and `Canada`/`Elsewhere` instead of `True`/`False`, so that the file is easy to read for anyone who opens it.

First, we need the number of nominations for each author. This is the same counting loop as in Week 6.

In [ ]:
nomination_counts = {}

for book in nominees:
    author = book["Author"]
    if author in nomination_counts:
        nomination_counts[author] = nomination_counts[author] + 1
    else:
        nomination_counts[author] = 1

print(nomination_counts["Michael Christie"])

Next, a small function that turns a region into `Canada`, `Elsewhere`, or `Unknown`. It uses the same test as our other calculations this week.

In [ ]:
def canada_or_elsewhere(region):
    """ Return "Canada" if region is in canadian_regions, "Unknown" if
        region is "-", and "Elsewhere" otherwise.
    """
    if region == "-":
        return "Unknown"
    elif region in canadian_regions:
        return "Canada"
    else:
        return "Elsewhere"


print(canada_or_elsewhere("ON"))
print(canada_or_elsewhere("Sri Lanka"))
print(canada_or_elsewhere("-"))

Now we write the file, one row per author. For each author we look up their cleaned regions in `born_regions` and `home_regions`, and replace any `"-"` with `Unknown`.

In [ ]:
with open("giller_authors.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Author", "Born", "Born region", "Born in Canada",
                     "Residence", "Lives region", "Lives in Canada", "Nominations"])

    for author in born_regions:
        # check this author has a residence too, so we don't get a KeyError
        if author in home_regions:
            born = birthplaces[author]
            parts = residences[author].split(" and ")
            residence = parts[0]

            born_region = born_regions[author]
            lives_region = home_regions[author]

            born_in_canada = canada_or_elsewhere(born_region)
            lives_in_canada = canada_or_elsewhere(lives_region)

            # replace "-" with "Unknown" so the file is easier to read
            if born == "-":
                born = "Unknown"
                born_region = "Unknown"
            if residence == "-":
                residence = "Unknown"
                lives_region = "Unknown"

            writer.writerow([author, born, born_region, born_in_canada,
                             residence, lives_region, lives_in_canada,
                             nomination_counts[author]])

print("Saved giller_authors.csv")

Open `giller_authors.csv` in JupyterLab's file browser to check the result. We'll use this file to start learning **pandas**, a library for working with tables.

## 7. Try it yourself

1. Spreadsheets typed by people contain mistakes: misspelled cities, or a city recorded in the wrong province. Print every birthplace and residence whose region is `ON` and check them against what you know (or can look up). Did you find any errors? Should we fix them in our code or in the spreadsheet?
2. Change the code that builds `home_regions` so that it uses the **last** place listed instead of the first. How do the four groups in section 4 change? Which rule do you think is fairer?
3. Use `born_regions` and `home_regions` to compare birthplaces and residences for Atlantic Canada (`NB`, `NL`, `NS`, `PE`): count how many nominees were born in an Atlantic province, and how many live in one. Do more nominees leave the Atlantic provinces than move there?
4. Repeat Steps 2 to 4 of section 5 using `home_regions` instead of `born_regions`, to compare where the nominees **live** with the population of each province. Which province gains the most nominees compared with where they were born? Which loses the most?